# 01 — Ingest

Fetch the raw data for **countries-income-disparity** from the web and land it in
`data/raw/` + DuckDB, untouched. Nothing here transforms the data — cleaning happens in `02-clean`.

## Sources (both free, no API key, public)

**1. World Bank — World Development Indicators (WDI), API v2.**
Base `https://api.worldbank.org/v2/country/all/indicator/{CODE}` (paginated JSON, `per_page=1000`,
`date=1960:2025`). Free, no key, generous limits. We pull the 14 indicators listed in
`config.yaml` (Gini, GDP/GNI per capita nominal + PPP, growth, poverty headcounts, life
expectancy, fertility, urbanization, unemployment, inflation, trade, internet). Each indicator is
fetched separately by `src.ingest.ingest_world_bank_indicator()` and cached to
`data/raw/wb_<code>.parquet`; `ingest_world_bank_all()` orchestrates all of them with a polite
1.5s delay between pages. License: **CC-BY 4.0**.

> ⚠️ The WDI response mixes **real countries with aggregates** ("World", "Euro area",
> income groups, regions). We keep them all in the raw long table here and separate them in
> `02-clean` by joining to the ISO reference. See SOURCES.md for the income-vs-consumption Gini
> comparability hazard.

**2. ISO 3166-1 country codes + UN geoscheme regions.**
`lukes/ISO-3166-Countries-with-Regional-Codes` CSV (public domain / CC0). Gives us alpha-2,
alpha-3, name, and UN region / sub-region — the crosswalk used in `02-clean` to (a) drop the WB
aggregates and (b) attach regions. Fetched by `src.ingest.ingest_iso_countries()` → `data/raw/iso_countries.csv`.

**3. World Bank Poverty & Inequality Platform (PIP) — the Gini welfare-metric flag.**
WDI's `SI.POV.GINI` gives a Gini value but NOT whether it's **income**- or **consumption**-based —
the single biggest cross-country comparability hazard (income Ginis run ~4.7 pts higher on
average). PIP is the underlying survey source and carries `welfare_type` per country/survey year,
so we pull it (`src.ingest.ingest_pip()` → `data/raw/pip_inequality.parquet`) to attach the metric
type to every Gini observation in `02-clean`. Free API, no key, CC-BY 4.0.

> Note: this ingest was originally run from the `src/` helpers directly; this notebook is the
> canonical narration of that run and re-runs the same functions (they skip re-download when the
> raw files already exist).

In [ ]:
import sys, os
from pathlib import Path

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config, ingest_iso_countries, ingest_world_bank_all, ingest_pip
from src.clean_quality import get_connection, load_to_duckdb, register_source

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## Source 1 — ISO 3166-1 country codes + regions

Downloads the ISO CSV to `data/raw/iso_countries.csv` (skips if already present) and loads it into
DuckDB as `countries`. This is reference data, not a statistical series.

In [ ]:
# ingest_iso_countries() now normalizes hyphenated headers (alpha-2 -> alpha_2) internally
df_iso = ingest_iso_countries(cfg, skip_existing=True)
load_to_duckdb(df_iso, 'countries', con)
register_source(
    con, 'countries',
    name='ISO 3166-1 Country Codes with UN Geoscheme',
    url='https://github.com/lukes/ISO-3166-Countries-with-Regional-Codes',
    license='Public domain / CC0',
    notes='alpha-2, alpha-3, numeric, name, UN region/sub-region. Join key to WDI = alpha_2. '
          'Reference crosswalk used to drop WB aggregates and attach regions in 02-clean.',
    retrieved='2026-08-27',
    methodology='Crosswalk of official ISO 3166-1 codes joined to the UN M49 geoscheme regions. '
                'Reference data, not statistical measurement.',
    series_breaks='None — reference table. ISO codes change rarely; region groupings follow the UN geoscheme.',
)
print(f'countries: {len(df_iso):,} rows')
df_iso.head()

## Source 2 — World Bank WDI indicators (long form)

Fetches all configured indicators (skips any already cached in `data/raw/`) and concatenates them
into one long table `indicators_long` (`country_code, country_name, year, value, indicator`).

**Known result:** of the 15 indicators in `config.yaml`, **14 return data**. `EN.ATM.CO2E.PC`
(CO2 per capita) came back empty from the API for this date range and is therefore absent from
`indicators_long` (and has no raw parquet). It stays in `config.yaml`/the catalog as a documented
gap; revisit if a later WDI edition repopulates it.

In [ ]:
df_long = ingest_world_bank_all(cfg, skip_existing=True)
load_to_duckdb(df_long, 'indicators_long', con)
register_source(
    con, 'indicators_long',
    name='World Bank World Development Indicators (14 indicators)',
    url='https://api.worldbank.org/v2/country/all/indicator/',
    license='CC-BY 4.0',
    notes='Long form: one row per country_code x year x indicator. Includes WB AGGREGATES '
          '(World, regions, income groups) alongside countries — separated in 02-clean via the '
          'ISO join. 14 of 15 configured indicators returned data (CO2 EN.ATM.CO2E.PC empty).',
    retrieved='2026-08-27',
    methodology='A COMPILATION, not primary collection. WB aggregates figures reported by national '
                'statistical offices; inequality/poverty indicators come from household surveys via '
                'the Poverty and Inequality Platform (PIP), run on each country\'s own schedule.',
    series_breaks='Gini is income-based (most high-income countries) vs consumption-based (most '
                  'low/middle-income) — NOT directly comparable (income ~4.7 pts higher on avg). '
                  'Surveys are irregular (the "year" is often the nearest survey year, not annual). '
                  'PPP rebasing (2017->2021 ICP) and PIP revisions shift values across WDI editions.',
)
print(f'indicators_long: {len(df_long):,} rows, {df_long["indicator"].nunique()} indicators')
df_long.head()

## Source 3 — World Bank PIP welfare metric (income vs consumption Gini)

Fetches the PIP inequality table (all countries, all years, national level) and loads it as
`pip_inequality`. The column that matters is **`welfare_type`** — `income` or `consumption` — which
tells us how each country's Gini was measured. WDI alone can't distinguish these, so a cross-country
Gini ranking that ignores it silently mixes methods (income Ginis run ~4.7 pts higher on average).

PIP `gini` is a **0–1 fraction**; `ingest_pip()` adds `gini_pct` (×100) so it lines up with WDI's
0–100 `SI.POV.GINI`. Country codes here are **ISO alpha-3** (e.g. `USA`), unlike the alpha-2 WDI
codes — `02-clean` joins on alpha-3. The pull is a single API call (~2,500 national survey rows).

In [ ]:
df_pip = ingest_pip(cfg, skip_existing=True)
load_to_duckdb(df_pip, 'pip_inequality', con)
register_source(
    con, 'pip_inequality',
    name='World Bank Poverty & Inequality Platform (PIP) — welfare metric',
    url='https://api.worldbank.org/pip/v1/pip', license='CC-BY 4.0',
    notes='One row per country (ISO alpha-3) x survey year. Key column welfare_type = income|'
          'consumption — the flag WDI SI.POV.GINI does not expose. gini is a 0-1 fraction; '
          'gini_pct = gini*100 to compare to WDI. Used in 02-clean to label each Gini by metric.',
    retrieved='2026-09-22',
    methodology='PIP estimates computed directly from national household income/consumption surveys, '
                'harmonized by the World Bank. welfare_type records the survey welfare aggregate.',
    series_breaks='EXISTS to resolve the income-vs-consumption break (income Gini ~4.7 pts higher). '
                  'survey_comparability / comparable_spell flag within-country vintage breaks; PPP '
                  'rebasing (2017->2021 ICP) and PIP revisions shift values across editions.',
)
print(f'pip_inequality: {len(df_pip):,} rows, {df_pip.country_code.nunique()} countries')
print('welfare_type mix:', df_pip.welfare_type.value_counts().to_dict())
df_pip[['country_code','country_name','reporting_year','welfare_type','gini','gini_pct']].head()

## Indicator catalog (code → readable column name)

A small lookup mapping the cryptic WB codes to the short column names used in the wide table and
throughout the project. Built from `config.yaml`'s indicator list. Loaded as `indicator_catalog`.
(Contains all 15 configured codes, including the currently-empty CO2 row, as documentation.)

In [ ]:
import pandas as pd

# short readable names, keyed by WB code
_short = {
    'SP.POP.TOTL': 'population', 'NY.GDP.PCAP.PP.CD': 'gdp_per_capita_ppp',
    'NY.GDP.PCAP.CD': 'gdp_per_capita_nominal', 'NY.GDP.MKTP.KD.ZG': 'gdp_growth_pct',
    'SI.POV.GINI': 'gini_index', 'SP.DYN.LE00.IN': 'life_expectancy',
    'SP.DYN.TFRT.IN': 'fertility_rate', 'SP.URB.TOTL.IN.ZS': 'urban_pct',
    'SL.UEM.TOTL.ZS': 'unemployment_pct', 'SI.POV.DDAY': 'poverty_215_pct',
    'SI.POV.LMIC': 'poverty_365_pct', 'FP.CPI.TOTL.ZG': 'inflation_pct',
    'NE.TRD.GNFS.ZS': 'trade_pct_gdp', 'IT.NET.USER.ZS': 'internet_pct',
    'EN.ATM.CO2E.PC': 'co2_per_capita',
}
cat = pd.DataFrame(
    [{'indicator_code': i['code'], 'column_name': _short[i['code']], 'description': i['name']}
     for i in cfg['sources']['world_bank']['indicators']]
)
load_to_duckdb(cat, 'indicator_catalog', con)
register_source(
    con, 'indicator_catalog',
    name='Indicator Code Reference', license='Public domain',
    notes='Maps WB indicator codes to readable column names. 15 codes; co2_per_capita is a '
          'documented gap (no data returned).',
    retrieved='2026-08-27',
    methodology='Lookup table derived from WDI indicator metadata + config.yaml.',
    series_breaks='Reference table; no break.',
)
cat

## Pivot to wide form → `countries_annual`

Pivot `indicators_long` to one row per `country_code x year` with a column per indicator (using the
catalog's short names). This is still **raw** (aggregates included, no region attached, no country
filtering) — it's just a more convenient shape. The country-level filtering + region join happens in
`02-clean`.

In [ ]:
code_to_col = dict(zip(cat['indicator_code'], cat['column_name']))
wide = (df_long
        .assign(column_name=lambda d: d['indicator'].map(code_to_col))
        .pivot_table(index=['country_code', 'country_name', 'year'],
                     columns='column_name', values='value', aggfunc='first')
        .reset_index())
wide.columns.name = None
load_to_duckdb(wide, 'countries_annual', con)
register_source(
    con, 'countries_annual',
    name='World Bank Indicators (wide format)',
    url='https://api.worldbank.org/v2/country/all/indicator/', license='CC-BY 4.0',
    notes='Pivoted wide form of indicators_long: one row per country_code (ISO alpha-2) x year, '
          'one column per indicator. STILL RAW — includes WB aggregates; no region/country filter yet.',
    retrieved='2026-08-27',
    methodology='Pivot of indicators_long; no transformation of values.',
    series_breaks='Inherits indicators_long breaks (income vs consumption Gini; irregular surveys; PPP rebasing).',
)
print(f'countries_annual: {len(wide):,} rows, years {wide.year.min()}-{wide.year.max()}')
wide.head()

## Quick inspection

Confirm the tables landed and eyeball the aggregate-vs-country mix we'll resolve next.

In [ ]:
for t in ['countries', 'indicators_long', 'indicator_catalog', 'countries_annual', 'pip_inequality', '_sources']:
    n = con.execute(f'select count(*) from "{t}"').fetchone()[0]
    print(f'{t:20s} {n:>10,} rows')

print('\nSample of codes that are WB AGGREGATES, not countries (dropped in 02-clean):')
print(con.execute("""
  select ca.country_code, ca.country_name
  from (select distinct country_code, country_name from countries_annual) ca
  left join countries r on ca.country_code = r.alpha_2
  where r.alpha_2 is null and ca.country_code in ('1W','XC','XD','Z4','OE')
  order by ca.country_code
""").df().to_string())

## Cleanup

DuckDB is single-writer — close the connection so other notebooks/scripts aren't blocked.

In [ ]:
con.close()
print('Connection closed.')

---
**Next:** `02-clean.ipynb` — drop the WB aggregates, fix the Namibia code, add Kosovo, attach
region/sub-region, run a quality report, and save the country-level interim dataset.